In [1]:
import pandas as pd
import numpy as np
import zipfile
import os
from pathlib import Path

In [2]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
import urllib.request

url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
zip_path = RAW_DIR / "complaints.csv.zip"

if not zip_path.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    print("Download complete.")
else:
    print("Dataset already exists.")

Dataset already exists.


In [4]:
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(RAW_DIR)

print("Files in raw folder:")
os.listdir(RAW_DIR)

Files in raw folder:


['complaints.csv.zip', 'complaints.csv']

In [6]:
csv_path = RAW_DIR / "complaints.csv"

sample_df = pd.read_csv(csv_path, nrows=10000)
sample_df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2024-11-04T18:11:36.000Z,Mortgage,FHA mortgage,Closing on a mortgage,Changes in loan terms during or after closing,NaN,NaN,"Better Mortgage, Inc.",MD,21117,NaN,Phone,2024-11-04T19:12:35.000Z,Closed with explanation,No,10684182
1,2025-03-02T12:43:51.000Z,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",This school XXXX XXXXXXXX XXXX XXXX XXXX was u...,NaN,MOHELA,FL,33309,NaN,Web,2025-04-07T16:59:46.000Z,Closed with explanation,No,12285693
2,2025-03-23T12:17:45.000Z,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Received bad information about your loan,I was placed in processing forbearance by XXXX...,NaN,MOHELA,CT,060XX,NaN,Web,2025-04-07T15:15:04.000Z,Closed with explanation,No,12661903
3,2025-04-07T21:53:35.000Z,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Received bad information about your loan,Type of Issue : Loan balance dispute Loan Serv...,NaN,MOHELA,MI,48307,NaN,Web,2025-04-07T22:04:49.000Z,Closed with explanation,No,12868665
4,2025-04-13T22:18:56.000Z,Student loan,Federal student loan servicing,Struggling to repay your loan,Problem with your payment plan,NaN,NaN,MOHELA,VA,22030,NaN,Web,2025-04-13T22:31:23.000Z,Closed with explanation,No,12966383


In [7]:
sample_df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
       'Company response to consumer', 'Timely response?', 'Complaint ID'],
      dtype='str')

In [9]:
row_count = 0

for chunk in pd.read_csv(csv_path, chunksize=100000, low_memory=False):
    row_count += len(chunk)

print(f"Total rows: {row_count:,}")

Total rows: 16,019,922


In [10]:
use_cols = [
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative",
    "Company public response",
    "Company",
    "State",
    "ZIP code",
    "Tags",
    "Consumer consent provided?",
    "Submitted via",
    "Date sent to company",
    "Company response to consumer",
    "Timely response?",
    "Consumer disputed?",
    "Complaint ID"
]

In [11]:
available_cols = sample_df.columns.tolist()
final_cols = [col for col in use_cols if col in available_cols]
final_cols

['Date received',
 'Product',
 'Sub-product',
 'Issue',
 'Sub-issue',
 'Consumer complaint narrative',
 'Company public response',
 'Company',
 'State',
 'ZIP code',
 'Tags',
 'Submitted via',
 'Date sent to company',
 'Company response to consumer',
 'Timely response?',
 'Complaint ID']

In [12]:
cleaned_chunks = []

for chunk in pd.read_csv(csv_path, usecols=final_cols, chunksize=100000, low_memory=False):
    chunk.columns = (
        chunk.columns
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("?", "", regex=False)
        .str.replace("-", "_")
    )
    
    chunk["date_received"] = pd.to_datetime(chunk["date_received"], errors="coerce")
    
    if "date_sent_to_company" in chunk.columns:
        chunk["date_sent_to_company"] = pd.to_datetime(chunk["date_sent_to_company"], errors="coerce")
    
    chunk = chunk.drop_duplicates()
    cleaned_chunks.append(chunk)

df = pd.concat(cleaned_chunks, ignore_index=True)
df.head()

,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id
0,2024-11-04 18:11:36+00:00,Mortgage,FHA mortgage,Closing on a mortgage,Changes in loan terms during or after closing,NaN,NaN,"Better Mortgage, Inc.",MD,21117,NaN,Phone,2024-11-04 19:12:35+00:00,Closed with explanation,No,10684182
1,2025-03-02 12:43:51+00:00,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",This school XXXX XXXXXXXX XXXX XXXX XXXX was u...,NaN,MOHELA,FL,33309,NaN,Web,2025-04-07 16:59:46+00:00,Closed with explanation,No,12285693
2,2025-03-23 12:17:45+00:00,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Received bad information about your loan,I was placed in processing forbearance by XXXX...,NaN,MOHELA,CT,060XX,NaN,Web,2025-04-07 15:15:04+00:00,Closed with explanation,No,12661903
3,2025-04-07 21:53:35+00:00,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Received bad information about your loan,Type of Issue : Loan balance dispute Loan Serv...,NaN,MOHELA,MI,48307,NaN,Web,2025-04-07 22:04:49+00:00,Closed with explanation,No,12868665
4,2025-04-13 22:18:56+00:00,Student loan,Federal student loan servicing,Struggling to repay your loan,Problem with your payment plan,NaN,NaN,MOHELA,VA,22030,NaN,Web,2025-04-13 22:31:23+00:00,Closed with explanation,No,12966383


In [13]:
# Convert both date columns again to make sure they are real datetime columns
df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce")

if "date_sent_to_company" in df.columns:
    df["date_sent_to_company"] = pd.to_datetime(df["date_sent_to_company"], errors="coerce")

    df["response_delay_days"] = (
        df["date_sent_to_company"] - df["date_received"]
    ).dt.days
else:
    df["response_delay_days"] = np.nan

df[["date_received", "date_sent_to_company", "response_delay_days"]].head()

,date_received,date_sent_to_company,response_delay_days
0,2024-11-04 18:11:36+00:00,2024-11-04 19:12:35+00:00,0.0
1,2025-03-02 12:43:51+00:00,2025-04-07 16:59:46+00:00,36.0
2,2025-03-23 12:17:45+00:00,2025-04-07 15:15:04+00:00,15.0
3,2025-04-07 21:53:35+00:00,2025-04-07 22:04:49+00:00,0.0
4,2025-04-13 22:18:56+00:00,2025-04-13 22:31:23+00:00,0.0


In [15]:
df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce", utc=True)
df["date_received"] = df["date_received"].dt.tz_localize(None)

df["year"] = df["date_received"].dt.year
df["month"] = df["date_received"].dt.month
df["year_month"] = df["date_received"].dt.to_period("M").astype(str)

df[["date_received", "year", "month", "year_month"]].head()

,date_received,year,month,year_month
0,2024-11-04 18:11:36,2024.0,11.0,2024-11
1,2025-03-02 12:43:51,2025.0,3.0,2025-03
2,2025-03-23 12:17:45,2025.0,3.0,2025-03
3,2025-04-07 21:53:35,2025.0,4.0,2025-04
4,2025-04-13 22:18:56,2025.0,4.0,2025-04


In [16]:
processed_path = PROCESSED_DIR / "complaints_cleaned.parquet"
df.to_parquet(processed_path, index=False)

print("Saved:", processed_path)

Saved: ../data/processed/complaints_cleaned.parquet
